## 2 - Data Ingestion with CREATE TABLE AS and COPY INTO

In this demonstration, we'll explore ingestion data from cloud storage into Delta tables with the `CREATE TABLE AS (CTAS)` and `COPY INTO` statements.

### Learning Objectives

By the end of this lesson, you should be able to:

- Use the CTAS statement with `read_files()` to ingest Parquet files into a Delta table.
- Use `COPY INTO` to incrementally load Parquet files from cloud object storage into a Delta table.

In [0]:
# from pyspark.sql import SparkSession
# from pyspark.sql.functions import col, lit
# import random

# spark = SparkSession.builder.getOrCreate()

# base_timestamp = 1592187000000

# for file_num in range(10):
#     data = []
#     start_index = 102350000 + file_num * 20
    
#     for i in range(20):
#         user_number = start_index + i
#         user_id = f"UA{str(user_number).zfill(12)}"
#         timestamp = base_timestamp + random.randint(0, 100000000)
#         email = None if random.random() < 0.7 else f"user{user_number}@example.com"
        
#         data.append((user_id, timestamp, email))
    
#     df = spark.createDataFrame(
#         data,
#         ["user_id", "user_first_touch_timestamp", "email"]
#     )
#     df.write.mode("append").parquet(
#     "/Volumes/azuredatabricks/v01/raw/users-historical/")

    


In [0]:
# /Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/

## B. Explore the Data Source Files

1. We'll create a table containing historical user data from Parquet files stored in the volume  
   `/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/` within Unity Catalog.

Use the `LIST` statement to view the files in this volume. Run the cell and review the results.

Notice the files in the name column begin with `part-`. This shows that this volume contains multiple Parquet files.

In [0]:
%sql
USE CATALOG azuredatabricks;
USE SCHEMA default;

In [0]:
spark.sql("LIST '/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/'").display()

In [0]:
%sql
SELECT *
FROM parquet.`/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/`
LIMIT 5;

## C. Batch Data Ingestion with CTAS and read_files()

The `CREATE TABLE AS` (CTAS) statement is used to create and populate tables using the results of a query. This allows you to create a table and load it with data in a single step, streamlining data ingestion workflows.

### Automatic Schema Inference for Parquet Files

Apache Parquet is a columnar storage format optimized for analytical queries. It includes embedded schema metadata (e.g., column names and data types), which enables automatic schema inference when creating tables from Parquet files. This eliminates the need for manual schema definitions and simplifies the process of converting Parquet files into Delta format by leveraging the built-in schema metadata.

## C1. CTAS with the `read_files()` Function

The code in the next cell creates a table using CTAS with the `read_files()` function.

The `read_files()` table-valued function (TVF) enables reading a variety of file formats and provides additional options for data ingestion.

1. First, let's explore the documentation for `read_files`. Complete the following steps:

   a. Navigate to the `read_files` documentation.  
   b. Scroll down and find the **Options** section. Take a moment to explore some of the features of `read_files`.  
   c. In the **Options** section, notice the variety of options available based on the file type.  
   d. Click on **parquet** and scroll through the available options.

> **NOTE:** The `read_files` function provides a wide range of capabilities and specific options for each file type. The previous method used to create a table only works if no additional options are required.

2. Use the `read_files()` function to query the same Parquet files located in `/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/`. The `LIMIT` clause limits the amount of rows during exploration and development.

- The first parameter in `read_files` is the path to the data.
- The `format => "parquet"` option specifies the file format.

The `read_files` function automatically detects the file format and infers a unified schema across all files. It also supports explicit schema definitions and `schemaHints`. For more details on schema inference capabilities, refer to the Schema inference documentation.

**NOTE:** A `_rescued_data` column is automatically included by default to capture any data that doesn’t match the inferred schema.

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/',
format => 'parquet'
)
LIMIT 5;


3. Next, let’s use `read_files()` with a CTAS statement to create the table `historical_users_bronze_ctas_rf`, then display the table.

Notice that the Parquet files were ingested to create a table (Delta by default).

In [0]:
%sql
DROP TABLE IF EXISTS historical_users_bronze_ctas_rf;

CREATE TABLE historical_users_bronze_ctas_rf
  SELECT *
  FROM read_files(
    '/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/',
    format => 'parquet'
  );

  SELECT *
  FROM historical_users_bronze_ctas_rf
  LIMIT 5;

4. Run the `DESCRIBE TABLE EXTENDED` statement to view column names, data types, and additional table metadata.

Review the results and notice the following:

- The table was created in your schema within the course catalog `azuredatabricks`.
- The `Type` row indicates that the table is **MANAGED**.
- The `Location` row shows the managed cloud storage location.
- The `Provider` row specifies that the table is a Delta table.

In [0]:
%sql
DESCRIBE DETAIL historical_users_bronze_ctas_rf;

In [0]:
%sql
DESCRIBE EXTENDED historical_users_bronze_ctas_rf;

In [0]:
%sql
DESCRIBE HISTORY historical_users_bronze_ctas_rf;

## Managed vs External Tables in Databricks

### Managed Tables
- Databricks manages both the data and metadata.
- Data is stored within Databricks’ managed storage.
- Dropping the table also deletes the data.
- Recommended for creating new tables.

### External Tables
- Databricks only manages the table metadata.
- Dropping the table does not delete the data.
- Supports multiple formats, including Delta Lake.
- Ideal for sharing data across platforms or using existing external data.

## C2. (BONUS) Python Ingestion
The code uses Python to ingest the parquet files.

In [0]:
# Read the parquet from the volume into a Spark dataframe
df = (spark
      .read
      .format("parquet")
      .load("/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/")
)
display(df)
      

In [0]:
# Write to the Dataframe to a Delta Table (overwrite if exists)
(df
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable('azuredatabricks.default.historical_users_bronze_python')
)

# Read and view the table
users_bronze_table = spark.read.table('historical_users_bronze_python')
display(users_bronze_table)

## D. Incremental Data Ingestion with `COPY INTO`

`COPY INTO` is a Databricks SQL command that allows you to load data from a file location into a Delta table. This operation is re-triable and idempotent, i.e. files in the source location that have already be loaded are skipped. This command is useful for when you need to load data into an existing Delta table.


## D1. Ingesting Parquet Files with COPY INTO

Using the same set of Parquet files as before, let's use `COPY INTO` to create our Bronze table again.

We will look at two examples:

1. Example 1: Common Schema Mismatch Error  
2. Example 2: Preemptively Handling Schema Evolution

## Example 1: Common Schema Mismatch Error

1. The cell below creates an empty table named `historical_users_bronze_ci` with a defined schema for only the `user_id` and `user_first_touch_timestamp` columns.

However, the Parquet files in `/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/` contain three columns:
- `user_id`
- `user_first_touch_timestamp`
- `email`

Run the cell below and review the error. You should see the `[COPY_INTO_SCHEMA_MISMATCH_WITH_TARGET_TABLE]` error. This error occurs because there is a schema mismatch: the Parquet files contain 3 columns, but the target table `historical_users_bronze_ci` only has 2 columns.

How can you handle this error?

In [0]:
%sql
-- DROP TABLE IF EXISTS historical_users_bronze_ci;

-- CREATE TABLE historical_users_bronze_ci(
--   user_id STRING,
--   user_first_touch_timestamp BIGINT
-- );

-- COPY INTO historical_users_bronze_ci
-- FROM '/Volumes/azuredatabricks/v01/raw/users-historical/'
-- FILEFORMAT = parquet;

2. We can fix this error by adding COPY_OPTIONS with the mergeSchema = 'true' option. When set to true, this option allows the schema to evolve based on the incoming data.

Run the next cell with the COPY_OPTIONS option added. You should notice that the Parquet files were successfully ingested into the table, with a total of 200 rows ingested.

In [0]:
%sql
COPY INTO historical_users_bronze_ci
FROM '/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/'
FILEFORMAT = parquet
COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
%sql
SELECT * FROM historical_users_bronze_ci;

## Example 2: Preemptively Handling Schema Evolution

1. Another way to ingest the same files into a Delta table is to start by creating an empty table named `historical_users_bronze_ci_no_schema`.

Then, add the `COPY_OPTIONS ('mergeSchema' = 'true')` option to enable schema evolution for the table.

Run the cell and confirm that 200 rows were added to the Delta table.

In [0]:
%sql
DROP TABLE IF EXISTS historical_users_bronze_ci_no_schema;

CREATE TABLE historical_users_bronze_ci_no_schema;

COPY INTO historical_users_bronze_ci_no_schema
FROM '/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/'
FILEFORMAT = parquet
COPY_OPTIONS ('mergeSchema' = 'true');

# D2. Idempotency (Incremental Ingestion)

COPY INTO tracks the files it has previously ingested. If the command is run again, no additional data is ingested because the files in the source directory haven't changed.

1. Let’s run the `COPY INTO` command again and check if any data is re-ingested into the table.

Run the cell and view the results. Notice that the values for `num_affected_rows`, `num_inserted_rows`, and `num_skipped_corrupt_files` are all 0 because the data has already been ingested into the Delta table.

> **NOTE:** If new files are added to the cloud storage location, `COPY INTO` will only ingest those files. Using `COPY INTO` is a great option if you want to run a job for incremental batch ingestion from cloud storage location without re-reading files that have already been loaded.

In [0]:
%sql
COPY INTO historical_users_bronze_ci_no_schema
FROM '/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/users-historical/'
FILEFORMAT = parquet
COPY_OPTIONS ('mergeSchema' = 'true');